In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
from time import time

from sklearn.pipeline import (
    Pipeline, make_pipeline
)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, train_test_split,
    KFold, StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier
)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import optuna

In [2]:
SEED = 67
WORK_DIR = Path(os.getcwd()).parent
DATA_DIR = WORK_DIR / 'data' / 'interim'

In [3]:
df = pd.read_csv(DATA_DIR / 'interim_hotel_bookings.csv', encoding='utf-8', parse_dates=['reservation_status_date'])
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,0,0,0,Transient,75.0,0,0,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304,0,0,Transient,75.0,0,0,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240,0,0,Transient,98.0,0,1,2015-07-03


In [4]:
y = df['is_canceled']
X = df.drop(columns=['is_canceled', 'reservation_status_date', 'booking_changes', 'days_in_waiting_list', 'assigned_room_type'])
X.shape, y.shape

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=SEED
)

X_sample, _, y_sample, _ = train_test_split(
    X_train, y_train, stratify=y_train, train_size=(X_train.shape[0]//8), test_size=0.2, random_state=SEED
)

In [5]:
numeric = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies',
    'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces',
    'total_of_special_requests'
]
categorical = X.columns.difference(numeric).tolist()

def cust_pipeline(model) -> Pipeline:
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=True))
    ])

    preprocessing = ColumnTransformer([
        ('numeric', num_pipe, numeric),
        ('categorical', cat_pipe, categorical)
    ])

    return Pipeline([('preprocessing', preprocessing), ('model', model)])

def models_params():
    return {
        'LogisticRegression': [LogisticRegression(max_iter=1000, random_state=SEED), 
                               {'model__C': [0.01, 0.1, 1, 10, 100]}],
        'KNeighborsClassifier': [KNeighborsClassifier(n_jobs=-1), 
                                 {'model__weights': ['uniform', 'distance'], 
                                  'model__n_neighbors': [15, 30, 50]}],
        'SVM': [SVC(probability=True, random_state=SEED), 
                {'model__C': [0.1, 1, 10], 
                 'model__kernel': ['poly', 'rbf']}],
        'MLPClassifier': [MLPClassifier(learning_rate='adaptive', max_iter=1000, early_stopping=True, random_state=SEED), 
                          {'model__hidden_layer_sizes': [(32,), (64,), (16, 32), (64, 32), (16, 16, 16)]}],
        'RandomForestClassifier': [RandomForestClassifier(n_jobs=-1, random_state=SEED),
                                   {'model__n_estimators': [100, 150, 250],
                                    'model__max_depth': [10, 16, 24]}],
        'CatBoostClassifier': [CatBoostClassifier(learning_rate=0.07, eval_metric='F1', iterations=400, verbose=False, thread_count=-1, random_state=SEED),
                               {'model__depth': [5, 8, 12]}],
        'LGBMClassifier': [LGBMClassifier(learning_rate=0.06, verbosity=-1, random_state=SEED, n_jobs=-1), 
                           {'model__n_estimators': [100, 150, 250],
                            'model__reg_lambda': [0.2, 0.4, 0.8],
                            'model__boosting_type': ['gbdt', 'dart']}],
        'AdaBoostClassifier': [AdaBoostClassifier(learning_rate=1.2, random_state=SEED),
                            {'model__n_estimators': [50, 100, 150, 250]}]
    }

In [6]:
(df['is_canceled'].value_counts() / df.shape[0] * 100).round(2)

is_canceled
0    62.96
1    37.04
Name: count, dtype: float64

In [7]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,...,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103886,0.007949,...,0.087118,0.137097,0.221124,74.828319,10.775157,2.321149,101.831122,0.062518,0.571363,2016-07-30 00:24:47.883407
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.380000,0.000000,0.000000,2014-10-17 00:00:00
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,7.000000,0.000000,0.000000,69.290000,0.000000,0.000000,2016-02-01 00:00:00
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,94.575000,0.000000,0.000000,2016-08-07 00:00:00
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,152.000000,0.000000,0.000000,126.000000,0.000000,1.000000,2017-02-08 00:00:00
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,...,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000,2017-09-14 00:00:00
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398555,0.097436,...,0.844336,1.497437,0.652306,107.141953,53.943884,17.594721,50.535790,0.245291,0.792798,NaN


In [8]:
fold_results = []
results = []
test_probabilities = {}
best_models = {}

def get_metrics(y_true, probability, threshold=0.5):
    predicted = (probability >= threshold).astype('int')

    return {
        'test_accuracy': accuracy_score(y_true, predicted),
        'test_precision': precision_score(y_true, predicted, zero_division=0),
        'test_recall': recall_score(y_true, predicted, zero_division=0),
        'test_f1': f1_score(y_true, predicted, zero_division=0),
        'test_roc_auc': roc_auc_score(y_true, probability),
        'test_pr_auc': average_precision_score(y_true, probability)
    }

def model_result(name: str) -> pd.DataFrame:
    model, param_grid = models_params()[name]
    cv_name, cv = ('StratifiedKFold', StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED))
    search = RandomizedSearchCV(
        cust_pipeline(model=model),
        param_distributions=param_grid,
        n_iter=4,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )
    search.fit(X_sample, y_sample)
    i = search.best_index_
    fold_results.append(
        {
            'model': name, 'cv': cv_name,
            'best_train_score': search.cv_results_['mean_train_score'][i],
            'best_test_score': search.cv_results_['mean_test_score'][i]
        }
    )
    p = search.predict_proba(X_test)[:, 1]
    test_probabilities[name] = p
    best_models[name] = search.best_estimator_
    results.append(
        {
            'model': name,
            'f1_best_score': search.best_score_,
            **get_metrics(y_test, p, threshold=0.42)
        }
    )  
    return pd.DataFrame(results)

In [9]:
def model_objective(trial):

    name = trial.suggest_categorical(
        'model',
        [
            'LogisticRegression',
            'KNeighborsClassifier',
            'SVM',
            'MLPClassifier',
            'RandomForestClassifier',
            'CatBoostClassifier',
            'LGBMClassifier',
            'AdaBoostClassifier'
        ]
    )

    if name == 'LogisticRegression':
        return cust_pipeline(LogisticRegression(
            C=trial.suggest_float('C', 1e-3, 100, log=True),
            max_iter=1000,
            random_state=SEED,
            n_jobs=-1
        ))

    if name == 'KNeighborsClassifier':
        return cust_pipeline(KNeighborsClassifier(
            weights=trial.suggest_categorical('weights', [
                'uniform', 'distance'
            ]),
            n_neighbors=trial.suggest_int('n_neighbors', 10, 150),
            n_jobs=-1
        ))

    if name == 'SVM':
        return cust_pipeline(SVC(
            C=trial.suggest_float('C', 1e-2, 100, log=True),
            kernel=trial.suggest_categorical('kernel', ['rbf', 'poly', 'linear']),
            max_iter=1000,
            random_state=SEED
        ))

    if name == 'MLPClassifier':
        return cust_pipeline(MLPClassifier(
            hidden_layer_sizes=trial.suggest_categorical('hidden_layer_sizes', [
                (32,), (64,), (16, 32), (64, 32), (16, 16, 16)
            ]),
            activation=trial.suggest_categorical('activation', [
                'relu', 'tanh', 'logistic'
            ]),
            alpha=trial.suggest_float('alpha', 1e-6, 1e-2, log=True),
            learning_rate='adaptive',
            early_stopping=True,
            random_state=SEED
        ))

    if name == 'RandomForestClassifier':
        return cust_pipeline(RandomForestClassifier(
            criterion=trial.suggest_categorical('criterion', [
                'gini', 'log_loss', 'entropy'
            ]),
            max_depth=trial.suggest_int('max_depth', 4, 20),
            n_estimators=trial.suggest_int('n_estimators', 50, 300),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 2, 50),
            random_state=SEED,
            n_jobs=-1
        ))

    if name == 'CatBoost':
        return cust_pipeline(CatBoostClassifier(
            iterations=trial.suggest_int('iteration', 200, 700),
            depth=trial.suggest_int('depth', 4, 20),
            learning_rate=trial.suggest_float(
                'learning_rate', 0.01, 0.2, log=True
            ),
            verbose=False,
            random_seed=SEED,
            thread_count=1,
        ))

    if name == 'LightGBM':
        return cust_pipeline(LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 500),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float(
                'learning_rate', 0.01, 0.2, log=True
            ),
            num_leaves=trial.suggest_int('num_leaves', 15, 80),
            random_state=SEED,
            n_jobs=1,
            verbosity=-1,
        ))

    return cust_pipeline(AdaBoostClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 400),
        learning_rate=trial.suggest_float(
            'learning_rate', 0.01, 2.0, log=True
        ),
        random_state=SEED,
    ))

def objective(trial):

    scores = cross_val_score(
        model_objective(trial),
        X_sample,
        y_sample,
        scoring='average_precision',
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
        n_jobs=-1
    )
    return scores.mean()

In [10]:
def final_model(params: dict):
    name = params['model']

    if name == 'LogisticRegression':
        return cust_pipeline(LogisticRegression(
            C=params['C'],
            max_iter=1000,
            random_state=SEED,
            n_jobs=-1
        ))
    
    if name == 'KNeighborsClassifier':
        return cust_pipeline(KNeighborsClassifier(
            weights=params['weights'],
            n_neighbors=params['n_neighbors'],
            n_jobs=-1
        ))
    
    if name == 'SVM':
        return cust_pipeline(SVC(
            C=params['C'],
            kernel=params['kernel'],
            max_iter=1000,
            random_state=SEED
        ))
    
    if name == 'MLPClassifier':
        return cust_pipeline(MLPClassifier(
            hidden_layer_sizes=params['hidden_layer_sizes'],
            activation=params['activation'],
            alpha=params['alpha'],
            learning_rate='adaptive',
            early_stopping=True,
            random_state=SEED
        ))
    
    if name == 'RandomForestClassifier':
        return cust_pipeline(RandomForestClassifier(
            criterion=params['criterion'],
            max_depth=params['max_depth'],
            n_estimators=params['n_estimators'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=SEED,
            n_jobs=-1
        ))
    
    if name == 'CatBoost':
        return cust_pipeline(CatBoostClassifier(
            iterations=params['iterations'],
            depth=params['depth'],
            learning_rate=params['learning_rate'],
            verbose=False,
            random_seed=SEED,
            thread_count=1,
        ))
    
    if name == 'LightGBM':
        return cust_pipeline(LGBMClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            num_leaves=params['num_leaves'],
            random_state=SEED,
            n_jobs=1,
            verbosity=-1,
        ))

    return cust_pipeline(AdaBoostClassifier(
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        random_state=SEED,
    ))

In [11]:
randGS_time0 = time()

In [12]:
best_log_reg = model_result('LogisticRegression')

In [13]:
best_log_reg

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.75605,0.748785,0.7524,0.897152,0.85322


In [14]:
best_KNN = model_result('KNeighborsClassifier')

In [15]:
best_KNN

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642


In [16]:
best_svm = model_result('SVM')

c:\Users\Maksim-Book\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [17]:
best_svm

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044


In [18]:
best_mlp = model_result('MLPClassifier')

In [19]:
best_mlp

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044
3,MLPClassifier,0.767402,0.834743,0.773229,0.783720,0.778439,0.920089,0.884240


In [20]:
best_forest = model_result('RandomForestClassifier')

In [21]:
best_forest

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044
3,MLPClassifier,0.767402,0.834743,0.773229,0.783720,0.778439,0.920089,0.884240
4,RandomForestClassifier,0.781054,0.862426,0.826444,0.795704,0.810783,0.935930,0.910679


In [22]:
best_catboost = model_result('CatBoostClassifier')

c:\Users\Maksim-Book\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [23]:
best_catboost

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044
3,MLPClassifier,0.767402,0.834743,0.773229,0.783720,0.778439,0.920089,0.884240
4,RandomForestClassifier,0.781054,0.862426,0.826444,0.795704,0.810783,0.935930,0.910679
5,CatBoostClassifier,0.797127,0.861881,0.807177,0.823968,0.815486,0.938421,0.911031


In [24]:
best_lgbm = model_result('LGBMClassifier')

In [25]:
best_lgbm

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044
3,MLPClassifier,0.767402,0.834743,0.773229,0.783720,0.778439,0.920089,0.884240
4,RandomForestClassifier,0.781054,0.862426,0.826444,0.795704,0.810783,0.935930,0.910679
5,CatBoostClassifier,0.797127,0.861881,0.807177,0.823968,0.815486,0.938421,0.911031
6,LGBMClassifier,0.794963,0.855390,0.796524,0.818768,0.807493,0.935575,0.905957


In [26]:
best_ada = model_result('AdaBoostClassifier')
randGS_time1 = time()

In [27]:
best_ada

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.731838,0.817447,0.756050,0.748785,0.752400,0.897152,0.853220
1,KNeighborsClassifier,0.728488,0.818410,0.744284,0.776597,0.760097,0.898247,0.866642
2,SVM,0.768332,0.842240,0.792646,0.777501,0.785001,0.915392,0.884044
3,MLPClassifier,0.767402,0.834743,0.773229,0.783720,0.778439,0.920089,0.884240
4,RandomForestClassifier,0.781054,0.862426,0.826444,0.795704,0.810783,0.935930,0.910679
5,CatBoostClassifier,0.797127,0.861881,0.807177,0.823968,0.815486,0.938421,0.911031
6,LGBMClassifier,0.794963,0.855390,0.796524,0.818768,0.807493,0.935575,0.905957
7,AdaBoostClassifier,0.740471,0.525965,0.438439,0.996043,0.608867,0.905687,0.865810


In [28]:
optuna_time0 = time()

study = optuna.create_study(sampler=optuna.samplers.TPESampler(), direction='maximize')
study.optimize(objective, n_trials=50)

optuna_time1 = time()

[I 2026-11-26 00:31:24,479] A new study created in memory with name: no-name-aa3110ac-899f-42b6-8393-f733aae7191d
[I 2026-11-26 00:31:26,757] Trial 0 finished with value: 0.8169570155305922 and parameters: {'model': 'LGBMClassifier', 'n_estimators': 168, 'learning_rate': 0.09095238804687095}. Best is trial 0 with value: 0.8169570155305922.
[I 2026-11-26 00:31:30,267] Trial 1 finished with value: 0.8326524082054504 and parameters: {'model': 'SVM', 'C': 0.011154935433279176, 'kernel': 'rbf'}. Best is trial 1 with value: 0.8326524082054504.
[I 2026-11-26 00:31:31,903] Trial 2 finished with value: 0.8554632622742049 and parameters: {'model': 'KNeighborsClassifier', 'weights': 'distance', 'n_neighbors': 80}. Best is trial 2 with value: 0.8554632622742049.
[I 2026-11-26 00:31:34,763] Trial 3 finished with value: 0.8381158436337653 and parameters: {'model': 'CatBoostClassifier', 'n_estimators': 206, 'learning_rate': 0.1448912326468254}. Best is trial 2 with value: 0.8554632622742049.
[I 2026-

In [29]:
print(f'Суммарное время обучения 8 моделей при помощи RandomizedGridSearchCV: {np.round(randGS_time1 - randGS_time0, 2)}\n')
print(f'Суммарное время обучения 8 моделей при помощи Optuna: {np.round(optuna_time1 - optuna_time0, 2)}')

Суммарное время обучения 8 моделей при помощи RandomizedGridSearchCV: 278.0

Суммарное время обучения 8 моделей при помощи Optuna: 123.92


In [30]:
study.best_value, study.best_params, study.best_trial

(0.9028505032836037,
 {'model': 'RandomForestClassifier',
  'criterion': 'log_loss',
  'max_depth': 20,
  'n_estimators': 246,
  'min_samples_leaf': 2},
 FrozenTrial(number=31, state=<TrialState.COMPLETE: 1>, values=[0.9028505032836037], datetime_start=datetime.datetime(2026, 11, 26, 0, 32, 40, 150151), datetime_complete=datetime.datetime(2026, 11, 26, 0, 32, 43, 350149), params={'model': 'RandomForestClassifier', 'criterion': 'log_loss', 'max_depth': 20, 'n_estimators': 246, 'min_samples_leaf': 2}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"model": "RandomForestClassifier"}'}, intermediate_values={}, distributions={'model': CategoricalDistribution(choices=('LogisticRegression', 'KNeighborsClassifier', 'SVM', 'MLPClassifier', 'RandomForestClassifier', 'CatBoostClassifier', 'LGBMClassifier', 'AdaBoostClassifier')), 'criterion': CategoricalDistribution(choices=('gini', 'log_loss', 'entropy')), 'max_depth': IntDistribution(high=20, log=False, low=4, step=1), 'n_estimators': 

In [31]:
best_trial = study.best_trial
best_model_name = study.best_params['model']
best_trial, best_model_name

(FrozenTrial(number=31, state=<TrialState.COMPLETE: 1>, values=[0.9028505032836037], datetime_start=datetime.datetime(2026, 11, 26, 0, 32, 40, 150151), datetime_complete=datetime.datetime(2026, 11, 26, 0, 32, 43, 350149), params={'model': 'RandomForestClassifier', 'criterion': 'log_loss', 'max_depth': 20, 'n_estimators': 246, 'min_samples_leaf': 2}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"model": "RandomForestClassifier"}'}, intermediate_values={}, distributions={'model': CategoricalDistribution(choices=('LogisticRegression', 'KNeighborsClassifier', 'SVM', 'MLPClassifier', 'RandomForestClassifier', 'CatBoostClassifier', 'LGBMClassifier', 'AdaBoostClassifier')), 'criterion': CategoricalDistribution(choices=('gini', 'log_loss', 'entropy')), 'max_depth': IntDistribution(high=20, log=False, low=4, step=1), 'n_estimators': IntDistribution(high=300, log=False, low=50, step=1), 'min_samples_leaf': IntDistribution(high=50, log=False, low=2, step=1)}, trial_id=31, value=None),


In [32]:
best_trial.params

{'model': 'RandomForestClassifier',
 'criterion': 'log_loss',
 'max_depth': 20,
 'n_estimators': 246,
 'min_samples_leaf': 2}

In [ ]:
best_model = final_model(best_trial.params)
best_model.fit(X_train, y_train)

In [ ]:
get_metrics(y_test, best_model.predict_proba(X_test))